# Case study: catching Telco churners when churners are the minority

*Session 9, block 3 practice. Author: course team, licence CC-BY-4.0.*

**The question.** 26.5 % of the IBM Telco customers left. A model trained on such data predicts "stays" most of the time and misses many churners. The retention team wants to find more of them. Is it better to change the *training data* (undersampling, oversampling, SMOTE), the *loss* (class weights), or only the *decision* (a threshold chosen from the costs, Session 8)?

- **Part A.** Strategies inside imbalanced-learn pipelines (undersampling, oversampling, SMOTENC, class weights) with a logistic regression and a random forest, compared with a lower threshold. Measures: precision, recall, F1 of the churn class, ROC AUC and PR AUC.
- **Part B.** The same strategies judged by the cost of a retention campaign (offer €100, lost customer €400, as in Session 8), against a threshold tuned by `TunedThresholdClassifierCV`.

Theory: [page 3](../theory/03-class-imbalance.md). The notebook downloads the Telco data from GitHub and runs in about two minutes.

In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.width", 120)
from imblearn.over_sampling import SMOTENC, RandomOverSampler
from imblearn.pipeline import make_pipeline
from imblearn.under_sampling import RandomUnderSampler
from sklearn.compose import make_column_transformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, average_precision_score, f1_score, make_scorer,
                             precision_recall_curve, precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import StratifiedKFold, TunedThresholdClassifierCV, cross_val_predict
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

# Part A: which customers will leave?

In [ ]:
URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
telco = pd.read_csv(URL)
telco["TotalCharges"] = pd.to_numeric(telco["TotalCharges"], errors="coerce").fillna(0)
y = (telco["Churn"] == "Yes").astype(int)
X = telco.drop(columns=["customerID", "Churn"])
cat_cols = X.select_dtypes("object").columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]
print(y.value_counts().to_dict(), "churn rate", round(y.mean(), 3))
cv = StratifiedKFold(5, shuffle=True, random_state=0)

SMOTENC needs to know which columns are categorical *after* preprocessing. We therefore encode the categories as integers (ordinal) and scale the numbers first, let SMOTENC interpolate only the numeric columns, and one-hot encode afterwards for the logistic regression. The forest can work with the ordinal codes directly.

In [ ]:
ordinal = make_column_transformer((OrdinalEncoder(), cat_cols), remainder=StandardScaler())   # categories first
cat_idx = list(range(len(cat_cols)))
onehot = make_column_transformer((OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_idx),
                                 remainder="passthrough")


def strategies(model, weighted):
    return {
        "no correction": make_pipeline(ordinal, onehot, model()),
        "undersampling": make_pipeline(ordinal, RandomUnderSampler(random_state=0), onehot, model()),
        "oversampling": make_pipeline(ordinal, RandomOverSampler(random_state=0), onehot, model()),
        "SMOTENC": make_pipeline(ordinal, SMOTENC(categorical_features=cat_idx, random_state=0), onehot, model()),
        "class weights": make_pipeline(ordinal, onehot, weighted()),
    }


models = {
    "logistic regression": strategies(lambda: LogisticRegression(max_iter=2000),
                                      lambda: LogisticRegression(max_iter=2000, class_weight="balanced")),
    "random forest": strategies(lambda: RandomForestClassifier(300, min_samples_leaf=5, random_state=0, n_jobs=-1),
                                lambda: RandomForestClassifier(300, min_samples_leaf=5, class_weight="balanced",
                                                               random_state=0, n_jobs=-1)),
}


def scores(proba, threshold=0.5):
    pred = (proba >= threshold).astype(int)
    return {"precision": precision_score(y, pred), "recall": recall_score(y, pred), "f1": f1_score(y, pred),
            "accuracy": accuracy_score(y, pred), "roc_auc": roc_auc_score(y, proba),
            "pr_auc": average_precision_score(y, proba)}


rows, probas = [], {}
for model_name, cands in models.items():
    for name, pipe in cands.items():
        proba = cross_val_predict(pipe, X, y, cv=cv, method="predict_proba")[:, 1]   # samplers: training folds only
        probas[(model_name, name)] = proba
        rows.append({"model": model_name, "strategy": name, **scores(proba)})
pd.DataFrame(rows).set_index(["model", "strategy"]).round(3)

### The alternative: keep the model, move the threshold

A threshold chosen on the cross-validated probabilities of the uncorrected model. Here we pick the threshold with the best F1, for illustration; in practice choose it from costs (Session 8) and on validation data only.

In [ ]:
for model_name in models:
    proba = probas[(model_name, "no correction")]
    prec, rec, thr = precision_recall_curve(y, proba)
    f1 = 2 * prec * rec / np.clip(prec + rec, 1e-9, None)
    best = thr[np.argmax(f1[:-1])]
    print(model_name, "best-F1 threshold", round(best, 2), {k: round(v, 3) for k, v in scores(proba, best).items()})

**Questions.**

1. Which strategies change ROC AUC and PR AUC, and which only move precision and recall? What does that tell you?
2. Compare the best correction with the uncorrected model at its best threshold. Is resampling worth the extra step?
3. A retention team can call 1,000 customers a month. Which number matters to them: F1, recall, or the precision among the 1,000 customers with the highest predicted risk? Compute the last one for two strategies.

In [ ]:
# precision among the 1,000 customers with the highest predicted churn risk
for key in [("logistic regression", "no correction"), ("logistic regression", "SMOTENC")]:
    top = np.argsort(-probas[key])[:1000]
    print(key, "precision in top 1,000:", round(y.iloc[top].mean(), 3))

# Part B: which strategy is cheapest for the retention team?

Session 8 priced the decision: an offer costs €100, a churner who is not reached costs €400. We compute the campaign cost of every strategy at its default threshold 0.5 from the cross-validated probabilities of Part A, and compare with the uncorrected logistic regression whose threshold `TunedThresholdClassifierCV` chooses from these costs inside each training fold (so its predictions are as honest as the others).

In [ ]:
OFFER, LOST = 100, 400


def campaign_cost(y_true, pred):
    y_true, pred = np.asarray(y_true), np.asarray(pred)
    return OFFER * (pred == 1).sum() + LOST * ((y_true == 1) & (pred == 0)).sum()


rows = [{"model": m, "strategy": s, "offers": int((p >= 0.5).sum()), "cost €": campaign_cost(y, (p >= 0.5).astype(int))}
        for (m, s), p in probas.items()]
tuned = TunedThresholdClassifierCV(models["logistic regression"]["no correction"],
                                   scoring=make_scorer(lambda yt, yp: -campaign_cost(yt, yp)), cv=5, random_state=0)
pred_tuned = cross_val_predict(tuned, X, y, cv=cv)              # threshold chosen inside each training fold
rows.append({"model": "logistic regression", "strategy": "no correction, cost-tuned threshold",
             "offers": int(pred_tuned.sum()), "cost €": campaign_cost(y, pred_tuned)})
cost_table = pd.DataFrame(rows).sort_values("cost €")
cost_table["cost per customer €"] = (cost_table["cost €"] / len(y)).round(1)
cost_table

In our run the uncorrected models at threshold 0.5 cost about €70–73 per customer; every correction and the cost-tuned threshold cost €62–65, within about €1 of each other. With these costs the optimal threshold (c / L = 0.25) is close to the churn rate (0.265), and balancing the classes moves a calibrated model's threshold to roughly that point.

**Questions.** (1) Which strategy is cheapest? By how much does it beat the uncorrected model at threshold 0.5? (2) The resampling strategies and class weights all move the model towards "churn". Why do they land close to the cost-tuned threshold here, and why would they not if a lost customer were worth €1,000? (3) Which solution would you hand to the retention team, and which one keeps probabilities that Finance can sum up into an expected number of churners (Session 8, calibration)?

## Exercises

1. In Part A, compare `RandomOverSampler(sampling_strategy=0.5)` (churners raised to half the loyal customers) with full balancing.
2. In Part B, set `LOST = 1000` and rerun the part. Which strategy wins now?
3. Write one sentence per strategy in the form "recall of churners rose from … to …, precision fell from … to …, the campaign cost changed by …".